In [1]:
import numpy as np
import matplotlib.pyplot as plt
import vaex as vx

from detprocess.core.didq import DIDQAnalysis, build_didq_row

In [2]:
RAW_PATH = '/sdata1/runs/run74/raw/exttrig_I2_D20260719_T145253'
SERIES = 'I2_D20260719_T145304'
THERMOMETER_CHANNELS = 'Mv6GaAs4pcBigFinsLeft,Mv6GaAs4pcBigFinsRight'
NB_EVENTS = 200
FCUTOFF_HZ = 200.0
LIST_OF_POLES = (2, 3)

## Load the raw dIdQ data

Each named channel is fitted as its own thermometer and stored under a key
of the form `<series>_<channel>`.

In [ ]:
analysis = DIDQAnalysis(verbose=True)
analysis.process_raw_data(
    raw_path=RAW_PATH,
    thermometer_channels=THERMOMETER_CHANNELS,
    series=SERIES,
    nb_events=NB_EVENTS,
)
keys = analysis.get_series_names()

for key in keys:
    metadata = analysis.get_didq_data(key)['metadata']
    print(f'{key}: {metadata["thermometer_channel"]}, '
          f'{analysis.get_didq_data(key)["drive_params"]}')

## Fit the ensemble mean

The fit runs on the averaged trace, weighted by the ensemble scatter, exactly
as dIdV is fitted.

In [4]:
analysis.dofit(
    list_of_poles=LIST_OF_POLES,
    fcutoff_hz=FCUTOFF_HZ,
)

INFO: fitting 2-pole model for series I2_D20260719_T145304 using 50 frequency bins
INFO: fitting 3-pole model for series I2_D20260719_T145304 using 50 frequency bins


## Inspect the fitted poles

Fall times are the reproducible output of the fit. For the three-pole model the
raw parameters are degenerate and should not be compared across series.

In [ ]:
for key in keys:
    for poles in LIST_OF_POLES:
        results = analysis.get_fit_results(key, poles)
        falltimes = np.asarray(results['falltimes'])
        falltimes = falltimes[np.argsort(np.abs(falltimes))[::-1]]
        print(f'{key} {poles}-pole falltimes [s]: '
              f'{np.array2string(falltimes, precision=5)}')
        print(f'    cost {results["cost"]:.4f}, '
              f'converged {results["fit_success"]}')

## Plot the measured transfer function against the fit

Only the odd harmonics of the drive frequency carry signal, so the plot shows
those bins alone.

In [ ]:
key = keys[0]
data = analysis.get_didq_data(key)
didvobj = data['didvobj']
frequency = didvobj._freq
transfer_function = didvobj._didvmean
driven = data['driven_mask']
positive = np.logical_and(frequency > 0, driven)

figure, axis = plt.subplots(figsize=(7, 4))
axis.loglog(
    frequency[positive],
    np.abs(transfer_function[positive]),
    marker='.',
    linestyle='none',
    label='measured',
)
axis.set_xlabel('Frequency [Hz]')
axis.set_ylabel('|dIdQ| [arb.]')
axis.set_title(f'dIdQ transfer function, {key}')
axis.legend()
figure.tight_layout()